# Practical 03: In-Memory Optimization via Dtype Downcasting & Deep Profiling

**UNIT 1: Introduction to Big Data and Large Dataset Handling**

**Problem Statement:** Write a Python program to demonstrate in-memory optimization of large datasets. Profile baseline memory usage using memory_usage(deep=True), downcast integer columns (int64 -> int32/int16), downcast float columns (float64 -> float32), and convert repetitive string columns to category type. Report the total percentage reduction in RAM footprint.

In [1]:
import numpy as np
import pandas as pd

rnd = np.random.default_rng(123)
n = 450_000
tbl = pd.DataFrame({
    "transaction_id": np.arange(1, n + 1, dtype="int64"),
    "customer_id": rnd.integers(1, 100000, n),
    "store_id": rnd.integers(1, 270, n),
    "quantity": rnd.integers(1, 17, n),
    "amount": np.round(rnd.gamma(2.0, 45.0, n), 2),
    "category": rnd.choice(["Electronics", "Clothing", "Groceries", "Furniture", "Beauty", "Toys"], n).astype(object),
    "payment_method": rnd.choice(["Credit Card", "Debit Card", "UPI", "Cash", "Net Banking"], n).astype(object),
    "city": rnd.choice(["Surat", "Mumbai", "Pune", "Nagpur", "Thane", "Rajkot", "Ahmedabad"], n).astype(object),
})
print(tbl.head())

def mb(frame):
    """Total deep memory of a DataFrame in MB."""
    return frame.memory_usage(deep=True).sum() / 1024 ** 2

start_mb = mb(tbl)
print(tbl.dtypes, "\n")
print("Memory per column (MB):")
print((tbl.memory_usage(deep=True) / 1024 ** 2).round(3))
print(f"\nBASELINE TOTAL: {start_mb:.2f} MB")

small = tbl.copy()
for col in small.select_dtypes(include="integer").columns:
    small[col] = pd.to_numeric(small[col], downcast="integer")

after_int = mb(small)
print(small.select_dtypes(include="integer").dtypes.to_dict())
print(f"After integer downcast: {after_int:.2f} MB")

for col in small.select_dtypes(include="float").columns:
    small[col] = pd.to_numeric(small[col], downcast="float")

after_float = mb(small)
print(small.select_dtypes(include="float").dtypes.to_dict())
print(f"After float downcast: {after_float:.2f} MB")

for col in small.select_dtypes(include=["object", "string"]).columns:
    if small[col].nunique() / len(small) < 0.35:
        small[col] = small[col].astype("category")

after_cat = mb(small)
print(small.dtypes)
print(f"\nAfter category conversion: {after_cat:.2f} MB")

reduction = (start_mb - after_cat) / start_mb * 100
print("=" * 45)
print(f"Baseline  : {start_mb:8.2f} MB")
print(f"Optimised : {after_cat:8.2f} MB")
print(f"REDUCTION : {reduction:8.2f} %")
print("=" * 45)

,transaction_id,customer_id,store_id,quantity,amount,category,payment_method,city
0,1,1545,220,11,24.60,Groceries,Cash,Nagpur
1,2,68235,210,8,170.66,Furniture,Debit Card,Surat
2,3,59295,52,7,131.95,Beauty,Debit Card,Pune
3,4,5383,44,3,164.84,Beauty,UPI,Surat
4,5,90908,135,9,188.65,Furniture,Credit Card,Rajkot


transaction_id      int64
customer_id         int64
store_id            int64
quantity            int64
amount            float64
category           object
payment_method     object
city               object
dtype: object 

Memory per column (MB):
Index              0.000
transaction_id     3.433
customer_id        3.433
store_id           3.433
quantity           3.433
amount             3.433
category          24.389
payment_method    24.377
city              23.540
dtype: float64

BASELINE TOTAL: 89.47 MB
{'transaction_id': dtype('int32'), 'customer_id': dtype('int32'), 'store_id': dtype('int16'), 'quantity': dtype('int8')}
After integer downcast: 80.46 MB
{'amount': dtype('float32')}
After float downcast: 78.74 MB
transaction_id       int32
customer_id          int32
store_id             int16
quantity              int8
amount             float32
category          category
payment_method    category
city              category
dtype: object

After category conversion: 7.73 MB
Baseli